# Screening Delegation Targets under Bargaining-Power Dependence
### A Synthetic-Data Proof-of-Concept

This notebook reproduces all results, tables, and figures.
**Design principle — generation–validation split:** the data-generating rules
(in `config.yaml`) are frozen independently of the screening procedure. A latent
true bargaining power `beta*` is generated as an answer key that the procedure
never observes; the procedure sees only noisy proxies.

Run order: config → generate → index → risk → screen → validate → replicate → figures.

In [1]:
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd
import screening as sc
cfg = sc.load_config('../config.yaml')
print('scenarios:', list(cfg['scenarios']))
print('target corr(b, beta*) band:',
      cfg['validation']['target_corr_beta_low'], '-', cfg['validation']['target_corr_beta_high'])

scenarios: ['oligopolistic', 'fragmented', 'intermediate']
target corr(b, beta*) band: 0.7 - 0.85


## 1. Single-scenario walkthrough
Generate one scenario, inspect the pipeline end to end.

In [2]:
df, m, ext_idx = sc.run_scenario(cfg, 'fragmented')
print('n targets:', len(df))
df[['domain','size','throughput','beta_star','b','risk','s','selected']].head()

n targets: 135


,domain,size,throughput,beta_star,b,risk,s,selected
0,E,1.273069,1.123556,0.192527,0.071062,0.690001,0.195461,False
1,E,1.008331,0.751231,0.275602,0.242338,0.513837,0.057546,False
2,E,1.066020,2.112723,0.369164,0.519406,0.558355,0.232977,False
3,E,1.432309,2.343653,0.527934,0.696512,0.981082,0.457678,False
4,E,1.197299,0.588912,0.377338,0.286952,0.448897,0.256275,False


### Validation metrics for this scenario
- `corr_b_beta`: convergent validity (b recovers latent beta*; beta* is unseen → no circularity)
- `corr_b_behavior`: criterion validity (b predicts a behavioral outcome not used to build b)
- `corr_b_s`: discriminant validity (should be ~0)
- `auc`: discrimination — does low risk rank targets near their individual U-trough optimum?
- `extreme_isolated`: was the planted super-supplier correctly excluded?

In [3]:
for k in ['corr_b_beta','corr_b_behavior','corr_b_s','auc','cliffs_delta',
          'silhouette','var_beta_star','extreme_isolated','extreme_b','extreme_risk']:
    print(f'{k:20s}: {m[k]}')

corr_b_beta         : 0.822539625872156
corr_b_behavior     : 0.6308758707113769
corr_b_s            : 0.016268117469559955
auc                 : 0.8450395083406497
cliffs_delta        : 0.5539872971065631
silhouette          : 0.1283602843425906
var_beta_star       : 0.03376848479088411
extreme_isolated    : True
extreme_b           : 0.7762412796078821
extreme_risk        : 1.157459555098871


## 2. Replications (100 seeds per scenario)
All five seed streams are offset jointly per replication; results are aggregated
to mean and 95% percentile intervals.

In [4]:
import run_replications as rr
tab, frames = rr.main()

scenario          fragmented  intermediate  oligopolistic
metric                                                   
auc                    0.831         0.828          0.823
cliffs_delta           0.451         0.429          0.413
corr_b_behavior        0.642         0.650          0.666
corr_b_beta            0.813         0.814          0.805
corr_b_s               0.012        -0.023          0.027
corr_b_share           0.848         0.846          0.843
extreme_isolated       1.000         1.000          1.000
silhouette             0.117         0.128          0.149
var_beta_star          0.038         0.043          0.052
variance_ratio         0.517         0.540          0.572

Saved tables to /home/claude/proj/results/tables


## 3. Figures
Grayscale, dpi 600, saved as PNG + PDF (no captions/titles) to `results/figures/`.
- Fig 1: capacity × bargaining-power positioning map (selected / isolated / extreme)
- Fig 2: AUC sensitivity over b-weight × noise-common-ratio grid (robustness)
- Fig 3: extreme-case isolation recall; structural & discrimination metrics by structure
- Fig 4: reliability-profile (rho) sensitivity of corr(b, beta*) and AUC, with pre-registered target band

In [5]:
import make_figures as mf
mf.fig1_positioning()
mf.fig2_sensitivity()
mf.fig3_isolation_and_discrimination()
mf.fig4_rho_sensitivity()
print('figures written to ../results/figures/')

saved fig1_positioning_map


saved fig2_sensitivity_heatmap


saved fig3_isolation_discrimination


saved fig4_rho_sensitivity
figures written to ../results/figures/


## 4. Key finding (data-driven)
Two separable propositions:
- **P1 (robustness):** procedure accuracy (AUC ~0.82) is invariant across all three
  market structures, across the b-weight x noise grid, and across joint reliability
  shifts.
- **P2 (conditional value):** discriminative value — the latent heterogeneity there
  is to exploit — rises monotonically with concentration
  (oligopolistic > intermediate > fragmented in Var(beta*) and silhouette).

The concentration ordering emerged **against** the prior expectation and is reported
unchanged, which is itself evidence the generation–validation split held. The planted
super-supplier is isolated in 100% of replications in every structure.